In [9]:
import numpy as np
import random 
import pandas as pd
import itertools

In [ ]:
random.seed(1)
number_of_decks = 2
sims = 1000000
deck_pen = 0.5

In [162]:
cards = ['A','2','3','4','5','6','7','8','9','10','J','Q','K']

card_values = {
    #'A' : [1, 11],
    'A' : [11],
    '2' : [2],
    '3' : [3],
    '4' : [4],
    '5' : [5],
    '6' : [6],
    '7' : [7],
    '8' : [8],
    '9' : [9],
    '10' : [10],
    'J' : [10],
    'Q' : [10],
    'K' : [10]
}

decks = []
for i in range(number_of_decks):
    for j in range(len(cards)):
        input = cards[j]
        decks.append(input)
        decks.append(input)
        decks.append(input)
        decks.append(input)

random.shuffle(decks)


In [163]:
dealer_face = cards.copy() 

player_cards = []
for i in range(len(cards)):
    for j in range(len(cards)):
        if j < i:
            continue
        else:
            player_cards.append([cards[i],cards[j]])

results = []
for i in dealer_face:
    for j in player_cards:
        results.append({'dealer_face': i, 'player_cards': ','.join(sorted(j))})

results_df = pd.DataFrame(results)

results_df['player_move_loss'] = 0
results_df['player_move_won'] = 0
results_df['player_move_total'] = 0

Moves:
first move only:
    double down - double the bet and take one more card
    split - seperate tow cards of the same calue into two hands
    surrender - forfeit the hand for half the bet back
    insurance - dealer face up card is ace but one-halg oringal bet that dealer has blackjack, pays 2 to 1
hit - take another cards
stand - take no more cards

Dealer 
hit until 17
hit soft 17 is a variation

In [164]:
def game_sim(number_of_decks = 2,
             sims = 100,
             deck_pen = 0.5):
    
    def deal_cards(current_card):
        player_cards = [decks[current_card], decks[current_card + 2]]
        dealer_cards = [decks[current_card + 1], decks[current_card + 3]]
        current_card = current_card + 4
        return player_cards, dealer_cards, current_card
    
    def next_move(first_move, current_card, card_list):
        if first_move == 'Y':
            #add in other moves here
            move = random.choice(['hit', 'stand'])
            if move == 'hit':
                return hit(current_card, card_list)
            else: 
                return stand(card_list, current_card)
        else:
            move = random.choice(['hit', 'stand'])
            if move == 'hit':
                return hit(current_card, card_list)
            else: 
                return stand(card_list, current_card)
    
    def hit(current_card, card_list):
        card_list.append(decks[current_card])
        current_card += 1
        if hand_values(card_list) > 21:
            return 'Bust', current_card
        else:
            return next_move('N', current_card, card_list)
    
    def stand(card_list, current_card):
        return hand_values(card_list), current_card
    
    def dealer_move(dealer_cards, current_card):

        while True:
            value = hand_values(dealer_cards)

            if value > 21:
                return 'Bust', current_card

            if value >= 17:   # TODO: adjust soft 17 later
                return value, current_card

            # Hit
            dealer_cards.append(decks[current_card])
            current_card += 1
         
    # this needs to deal with aces
    def hand_values(card_list):
        output = 0
        for i in card_list:
            output += card_values[i][0]
        return output
    
    current_card = 0
    while current_card < round(len(decks)*deck_pen):
        first_move = 'Y'
        player_cards, dealer_cards, current_card = deal_cards(current_card)
        player_result, current_card = next_move(first_move, current_card, player_cards.copy())
        print('Player Result')
        print(player_result)
        print('Current Card')
        print(current_card)

        if player_result == 'Bust':
            results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'player_move_loss' ] += 1
            results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'player_move_total' ] += 1
        else:
            dealer_result, current_card = dealer_move(dealer_cards.copy(), current_card)
            print('Dealer Result')
            print(dealer_result)
            if dealer_result == 'Bust':
                results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'player_move_won' ] += 1
                results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'player_move_total' ] += 1
            elif dealer_result > player_result:
                results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'player_move_loss' ] += 1
                results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'player_move_total' ] += 1
            else:
                results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'player_move_won' ] += 1
                results_df.loc[(results_df['dealer_face'] == dealer_cards[0]) & (results_df['player_cards'] == ','.join(sorted(player_cards[:2]))), 'player_move_total' ] += 1


In [165]:
game_sim()

Player Result
20
Current Card
4
Dealer Result
19
Player Result
15
Current Card
8
Dealer Result
17
Player Result
Bust
Current Card
16
Player Result
Bust
Current Card
22
Player Result
20
Current Card
26
Dealer Result
21
Player Result
18
Current Card
30
Dealer Result
17
Player Result
15
Current Card
34
Dealer Result
Bust
Player Result
14
Current Card
39
Dealer Result
Bust
Player Result
Bust
Current Card
45
Player Result
19
Current Card
49
Dealer Result
Bust
Player Result
12
Current Card
54
Dealer Result
Bust


In [166]:
results_df[results_df['player_move_total'] > 0]

,dealer_face,player_cards,player_move_loss,player_move_won,player_move_total
84,A,"10,K",1,0,1
329,4,"6,7",1,0,1
463,6,"9,A",1,0,1
481,6,"3,4",1,0,1
598,7,"5,J",1,0,1
599,7,"5,Q",0,1,1
618,7,"10,8",0,1,1
772,9,"4,Q",0,1,1
811,9,"10,Q",0,1,1
1041,Q,"4,8",0,1,1
